In [ ]:
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
# Load direct from split folder
train_df = pd.read_csv('../data/split/train/train.csv')
test_df  = pd.read_csv('../data/split/test/test.csv')


In [ ]:
for df in [train_df, test_df]:
    df["area_location"] = df["area"] * df.groupby("address")["price"].transform("median")

# Chọn feature
features = ["area", "bedrooms", "bathrooms", "area_location"]
target_col = "price"

X_train = train_df[features]
y_train = train_df[target_col]
X_test  = test_df[features]
y_test  = test_df[target_col]



In [ ]:
# X, y
X_train = train_df[features].values
y_train = train_df["price"]

X_test = test_df[features].values
y_test = test_df["price"]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

y_train = y_train.values.reshape(-1, 1)
y_test = y_test.values.reshape(-1, 1)

In [ ]:
# Convert to tensor
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_test  = torch.tensor(X_test, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
# Model
model = torch.nn.Linear(X_train.shape[1], 1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = torch.nn.MSELoss()

In [ ]:
# Train
for epoch in range(10000):
    optimizer.zero_grad()
    pred = model(X_train)
    loss = loss_fn(pred, y_train)
    loss.backward()
    optimizer.step()
    if epoch % 1000 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 146.47140502929688
Epoch 1000, Loss: 43.33148956298828
Epoch 2000, Loss: 40.978702545166016
Epoch 3000, Loss: 40.974334716796875
Epoch 4000, Loss: 40.974342346191406
Epoch 5000, Loss: 40.974334716796875
Epoch 6000, Loss: 40.974334716796875
Epoch 7000, Loss: 40.974342346191406
Epoch 8000, Loss: 40.974342346191406
Epoch 9000, Loss: 40.974334716796875


In [ ]:
# Print weights and bias
print("\n=== MODEL PARAMETERS ===")
print("Weights:", model.weight.detach().numpy())
print("Bias:", model.bias.detach().numpy())

# Evaluate
y_pred = model(X_test).detach().numpy()
print("\n=== EVALUATION ===")
print("R²:", r2_score(y_test, y_pred))
print("RMSE:", (mean_squared_error(y_test, y_pred)) ** 0.5)


=== MODEL PARAMETERS ===
Weights: [[ 0.72640157 -0.11883964  1.6152039   5.7384186 ]]
Bias: [8.433246]

=== EVALUATION ===
R²: 0.649334192276001
RMSE: 6.766999208644473
